# Mean Reversion Strategy Backtest — INTC vs KO

**Objective:** test a mean-reversion trading strategy against a buy-and-hold benchmark on two large-cap stocks (Intel and Coca-Cola), comparing risk-adjusted return and equity curve behaviour.

**Universe:** INTC, KO
**Period:** 2022-01-01 to present, daily data (Yahoo Finance)
**Metric:** Sharpe ratio (strategy vs buy-and-hold) and equity curve comparison

In [7]:
import pandas as pd
import yfinance as yf
import numpy as np
from pipeline import backtest_mean_reversion
import plotly.graph_objects as go

## Data & backtest

Download daily OHLCV data for INTC and KO from Yahoo Finance (2022-01-01 onward), flatten the multi-index columns returned by `yfinance`, then run `backtest_mean_reversion()` on each ticker. The function returns an equity curve for the strategy alongside a buy-and-hold benchmark, and prints the Sharpe ratio for both.

In [6]:
# Download data
intc = yf.download('INTC', start='2022-01-01')
intc.columns = intc.columns.get_level_values(0)

ko = yf.download('KO', start='2022-01-01')
ko.columns = ko.columns.get_level_values(0)

# Run strategy on both instantly
intc_results = backtest_mean_reversion(intc)
ko_results = backtest_mean_reversion(ko)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Mean Reversion Strategy Sharpe Ratio: 0.50
Buy and Hold Sharpe Ratio: 0.52
Mean Reversion Strategy Sharpe Ratio: 0.21
Buy and Hold Sharpe Ratio: 0.77


## INTC — strategy vs buy & hold

In [11]:
fig = go.Figure()
# Add traces for INTC
fig.add_trace(go.Scatter(x=intc_results.index, y=intc_results['Equity_Curve'], mode='lines', name='INTC Equity Curve'))
fig.add_trace(go.Scatter(x=intc_results.index, y=intc_results['Buy_Hold'], mode='lines', name='INTC Buy & Hold'))

fig.update_layout(title='INTC Mean Reversion Strategy vs Buy & Hold',template='plotly_dark', xaxis_title='Date', yaxis_title='Portfolio Value')
fig.show()

**Sharpe ratio:** strategy 0.50 vs buy & hold 0.52 — essentially tied on a risk-adjusted basis.

**Equity curve:** total returns were comparable, but the strategy curve was noticeably smoother. It wasn't dragged around by the July spike that shows up in the buy-and-hold curve, which lowers volatility/drawdown risk without sacrificing return.

**Takeaway:** on INTC, mean reversion matched buy-and-hold on return and Sharpe, but delivered a smoother path — a win on risk even though the headline Sharpe numbers look tied.

## KO — strategy vs buy & hold

In [12]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=ko_results.index, y=ko_results['Equity_Curve'], mode='lines', name='KO Equity Curve'))
fig.add_trace(go.Scatter(x=ko_results.index, y=ko_results['Buy_Hold'], mode='lines', name='KO Buy & Hold'))

fig.update_layout(title='KO Mean Reversion Strategy vs Buy & Hold',template='plotly_dark', xaxis_title='Date', yaxis_title='Portfolio Value')
fig.show()

**Sharpe ratio:** strategy 0.21 vs buy & hold 0.77 — buy-and-hold is clearly stronger on a risk-adjusted basis here.

**Equity curve:** the strategy curve was smooth (low volatility), but that smoothness came at the cost of return — it underperformed buy-and-hold, so lower volatility wasn't enough to offset the return gap.

**Takeaway:** on KO, mean reversion reduced volatility but didn't earn its keep — buy-and-hold delivered both the higher return and the higher Sharpe ratio.

## Overall conclusion

- **INTC:** Sharpe ratios essentially tied (0.50 vs 0.52), but the strategy's smoother curve — undisturbed by the July spike — makes it the better risk-adjusted choice.
- **KO:** buy-and-hold wins on both return and Sharpe (0.77 vs 0.21); the strategy's smoothness didn't translate into competitive performance.
- **Net read:** mean reversion isn't a universal edge — it worked well on INTC (smoother path, same return) but underperformed on KO (smoother path, lower return). Worth checking whether KO's steadier long-term uptrend (a low-mean-reversion regime) explains the underperformance, versus INTC's choppier, more range-bound price action.